# LAX Daily-High Climatology — Layer 1 Exploration

**Goal:** build the climatology prior (Layer 1 of the pipeline) and validate it against today's NWS forecast.

**What the climatology gives us:** for any calendar date, an empirical probability distribution over the daily high at LAX based on observations within ±15 days of that day-of-year across 20 years.

**What it does NOT give us:** any current information. If a marine layer is sitting over LA today, climatology doesn't know. Layer 2 (NWS forecast) and Layer 3 (HRRR) inject that current state. We'll compare climatology vs NWS at the end to show the gap.

### Data sources

| Source | Role | Why |
|---|---|---|
| **NCEI Daily Summaries** | Training | 20 years of QC'd history, easy to fetch, near-identical to CLI |
| **NWS Daily Climate Report (CLI)** | **Settlement / ground truth** | This is what Kalshi reads at expiration. Authoritative. |
| **NWS forecast (api.weather.gov)** | Layer 2 input | The NWS office's own forecast; baseline prediction |

NCEI and CLI normally agree because both ultimately read the LAX ASOS, but they're separate artifacts and the CLI is the canonical source for contract resolution. Section 7 of this notebook cross-checks them.

---

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from lax_forecast.data import load_lax_history
from lax_forecast.climatology import build_climatology_from_loaded
from lax_forecast.nws import get_daily_high
from lax_forecast.nws_climate_report import get_report_for_date, get_latest_report

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

## 1. Load historical observations

In [ ]:
result = load_lax_history()
df = result.df
print(f'Rows: {len(df):,}   Range: {df.index.min().date()} → {df.index.max().date()}')
print(f'Dropped {result.rows_dropped_quality} QC failures, {result.rows_dropped_missing} missing TMAX')
df.head()

In [ ]:
df['tmax_f'].describe().to_frame().T

## 2. Annual cycle — what "average" looks like by day-of-year

In [ ]:
by_doy = df.assign(doy=df.index.dayofyear).groupby('doy')['tmax_f'].agg(
    mean='mean', std='std', q05=lambda s: s.quantile(0.05),
    q25=lambda s: s.quantile(0.25), q50=lambda s: s.quantile(0.50),
    q75=lambda s: s.quantile(0.75), q95=lambda s: s.quantile(0.95)
)

fig, ax = plt.subplots(figsize=(11, 5))
ax.fill_between(by_doy.index, by_doy['q05'], by_doy['q95'], alpha=0.15, label='5–95th pctile')
ax.fill_between(by_doy.index, by_doy['q25'], by_doy['q75'], alpha=0.25, label='25–75th pctile')
ax.plot(by_doy.index, by_doy['q50'], lw=1.5, label='median')
ax.set_xlabel('Day of year')
ax.set_ylabel('TMAX (°F)')
ax.set_title('LAX daily-high climatology, 2006–present')
ax.set_xlim(1, 366)
ax.legend(loc='upper left')
plt.tight_layout()

Notice the right-skew between June–November: q50→q95 is much wider than q05→q50. That asymmetry is **Santa Ana days** — rare but extreme heat events that pull the upper tail far above the seasonal median. The lower tail (marine layer days) is tighter because there's a hard floor on how cold LA gets in summer.

For trading, this means: in summer, *out-of-the-money strikes above the median are systematically underpriced* if the market uses symmetric distributions around the NWS forecast.

## 3. Build the climatology prior

In [ ]:
# Two variants to compare: uniform across years vs. recency-weighted.
clim_uniform = build_climatology_from_loaded(df, window_days=15)
clim_recent  = build_climatology_from_loaded(df, window_days=15, recency_halflife_years=10)

target_date = pd.Timestamp.today().normalize()
print(f'Target date: {target_date.date()}  (DOY {target_date.dayofyear})')

for name, clim in [('uniform 20yr', clim_uniform), ('10yr-halflife', clim_recent)]:
    d = clim.distribution(target_date)
    print(f'  {name:>14}: mean={d.mean:5.2f}°F  std={d.std:4.2f}°F  '
          f'q05={d.quantile(0.05):.0f}  q50={d.quantile(0.5):.0f}  q95={d.quantile(0.95):.0f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
for name, clim in [('uniform 20yr', clim_uniform), ('10yr-halflife', clim_recent)]:
    d = clim.distribution(target_date)
    ax.bar(d.temps_f, d.probs, alpha=0.5, label=name, width=0.85)
ax.set_xlabel(f'TMAX (°F) on {target_date.date()}')
ax.set_ylabel('Probability')
ax.set_title('Climatology prior — empirical distribution from historical observations')
ax.legend()
plt.tight_layout()

## 4. Fetch today's NWS forecast — Layer 2 baseline

In [ ]:
nws = get_daily_high()  # default = today
print(f'NWS forecast for {nws.target_date}:')
print(f'  high = {nws.high_f}°F')
print(f'  short = {nws.short_forecast!r}')
print(f'  detail = {nws.detailed_forecast}')

## 5. Climatology vs NWS — where the gap is

Climatology is symmetric around the seasonal mean. NWS conditions on today's atmospheric state. If they disagree by more than 2–3°F, it's almost always because NWS sees something climatology can't (marine layer, Santa Ana setup, etc.).

Trading climatology vs trading NWS-conditioned distributions is the difference between random walk and skill.

In [ ]:
clim_dist = clim_recent.distribution(target_date)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(clim_dist.temps_f, clim_dist.probs, alpha=0.4, label=f'Climatology (mean {clim_dist.mean:.1f}°F)', width=0.9)
ax.axvline(nws.high_f, color='red', lw=2.5, label=f'NWS forecast ({nws.high_f}°F)')
ax.set_xlabel(f'TMAX (°F) on {target_date.date()}')
ax.set_ylabel('Probability')
ax.set_title(f'Climatology prior vs. NWS forecast — {target_date.date()}')
ax.legend()
plt.tight_layout()

## 6. Convert to Kalshi strike probabilities

This is what we eventually compare against the order book. For a sweep of strikes around the climatology mean, we compute P(payout) for the three Kalshi contract types: *greater than*, *less than*, *between*.

In [ ]:
mean = clim_dist.mean
strikes = np.arange(int(mean) - 8, int(mean) + 9)
table = pd.DataFrame({
    'strike_°F': strikes,
    'P(>strike)': [clim_dist.p_greater_than(s) for s in strikes],
    'P(<strike)': [clim_dist.p_less_than(s) for s in strikes],
}).set_index('strike_°F')
table

In [ ]:
# 'Between' bucket prices for 3°F-wide brackets (typical Kalshi listing).
buckets = [(int(mean) + d, int(mean) + d + 2) for d in range(-9, 9, 3)]
bucket_table = pd.DataFrame({
    'bucket': [f'{lo}–{hi}°F' for lo, hi in buckets],
    'P(between)': [clim_dist.p_between(lo, hi) for lo, hi in buckets],
})
print(f'Climatology mean = {mean:.1f}°F  (NWS = {nws.high_f}°F)')
bucket_table

## 7. Cross-check: NCEI archive vs NWS Daily Climate Report

Kalshi settles on the NWS Daily Climate Report (`CLI` text bulletin), not the NCEI archive. We train on NCEI for convenience (clean tabular history) but need to verify the two sources agree on the dates Kalshi cares about. If they ever diverge, we should believe CLI for label purposes.

In [ ]:
# Compare the last several days for which both sources have data.
# NCEI lags by 1-3 days; CLI is published the morning after the measurement day.
n_compare = 5
rows = []
for d in pd.date_range(end=df.index.max(), periods=n_compare):
    ncei_max = int(df.loc[d, 'tmax_f'])
    try:
        cli = get_report_for_date(d.date(), search_limit=30)
        rows.append({
            'date': d.date(),
            'ncei_tmax_f': ncei_max,
            'cli_tmax_f': cli.high_f,
            'agree': '✓' if cli.high_f == ncei_max else '✗ DIFF',
            'cli_issued': cli.issuance_time.isoformat(timespec='minutes'),
        })
    except LookupError:
        rows.append({
            'date': d.date(),
            'ncei_tmax_f': ncei_max,
            'cli_tmax_f': None,
            'agree': 'CLI too old in product list',
            'cli_issued': None,
        })

pd.DataFrame(rows).set_index('date')

## What's next

Climatology alone is a deliberately weak baseline. Real edge comes from Layer 2 (NWS forecast + bias correction) and Layer 3 (HRRR post-processing). Specifically:

1. **Pull the NWS NDFD archive** so we can compute historical NWS forecast errors. The residual distribution → NWS bias correction + calibrated variance.
2. **Add HRRR forecasts** for marine layer detection (the LAX-specific edge).
3. **Backtest each layer's calibration** with Brier score and log-loss against held-out actuals.
4. **Plug into a Kalshi orderbook fetcher** to identify mispricings in real time.